# Inwiefern besteht ein Zusammenhang zwischen der Brutto CO2 Einspeisung, Trockenheit, Pflanzenmenge und der Bodentemperatur in Deutschland?

### bezogen auf die Hitzewelle 2018 in Deutschland

Was brauchen wir?

Erstmal: GPP Vergleich mit Trockenheit. Dafür: zwei Karten. Graph (Durschnittliche Trockenheit in Deutschland + Durschnittlicher GPP in Deutschland)

Plan::: Daten für GPP, Trockenheit, Bodentemperatur download. => auf gleiches Raster interpolieren. => alle drei auf nem Line Graph plotten

- CO2: CLMS_GPP_GLOBAL_300M_10DAILY_V2 2013-present
- Feuchtigkeit: CLMS_SSM_EUROPE_1KM_DAILY_V1 2014-present
- Temperatur: CLMS_LST_TCI_GLOBAL_3KM_10DAILY_V3 2016-present

=> 2016-present?

In [37]:
GPP_ID = "CLMS_GPP_GLOBAL_300M_10DAILY_V2"
SSM_ID = "CLMS_SSM_EUROPE_1KM_DAILY_V1"
LST_ID = "CLMS_LST_TCI_GLOBAL_3KM_10DAILY_V3"

In [38]:
spatial_extent = {
    'west': 5,
    'south': 46,
    'east': 15.5,
    'north': 56,
}

In [39]:
temporal_extent = ["2017-01-01", "2019-12-31"]

Imports

In [ ]:
import folium
import geopandas as gpd
import json
import leafmap
import math
import matplotlib.pyplot as plt
import os
import openeo
import pyproj
import rasterio

from rasterio.plot import show
from shapely.geometry import Polygon
from datetime import datetime

Authentification

In [41]:
from openeo.rest.auth.config import RefreshTokenStore

connection = openeo.connect("openeofed.dataspace.copernicus.eu")
connection.authenticate_oidc()
print(connection.describe_account())

Authenticated using refresh token.
{'info': {'oidc_userinfo': {'email': 'moritz.fechte@icloud.com', 'email_verified': True, 'family_name': 'Fe', 'given_name': 'Mo', 'name': 'Mo Fe', 'preferred_username': 'moritz.fechte@icloud.com', 'sub': '01b88e4b-35a5-4da2-8e10-edc6a6ce408f'}}, 'name': 'Mo Fe', 'user_id': '01b88e4b-35a5-4da2-8e10-edc6a6ce408f'}


### Datacube Preprocessing Methoden

Deutschland ausschneiden (germany.geojson Datei notwendig)

In [42]:
def get_german_geometry():
    germany = gpd.read_file("germany.geojson")
    return germany.geometry.iloc[0].__geo_interface__

def mask_germany(datacube, german_geometry):
    return datacube.mask_polygon(
        german_geometry,
        srs="EPSG:4326"
    )

Zeit auf 10 Tage Intervall reduzieren

In [43]:
def reduce_to_decad(datacube):
    return datacube.aggregate_temporal_period(
        period="dekad",
        reducer="mean"
    )

Auflösung auf 1km reduzieren (resamplen)

In [44]:
def align_to_other_cube(grid_providing_cube, cube):
    return cube.resample_cube_spatial(
        target=grid_providing_cube,
        method="average"
    )

In [45]:
def get_mean(datacube, german_geometry):
    return datacube.aggregate_spatial(
        geometries=german_geometry,
        reducer='mean',
    )

### GPP Download

In [46]:
def load_datacubes():
    gpp = connection.load_collection(
        GPP_ID,
        spatial_extent=spatial_extent,
        temporal_extent = temporal_extent,
        bands=["gpp"],
    )

    ssm = connection.load_collection(
        SSM_ID,
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=["ssm"],
    )

    lst = connection.load_collection(
        LST_ID,
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=["tci_median"],
    )

    return gpp, ssm, lst

In [47]:
# preprocessing execution
def preprocess_maps(gpp, ssm, lst):
    # select bands
    gpp = gpp.band('gpp')
    ssm = ssm.band('ssm')
    lst = lst.band('tci_median')

    gpp = reduce_to_decad(gpp)
    gpp = align_to_other_cube(ssm, gpp)
    gpp = mask_germany(gpp)

    ssm = reduce_to_decad(ssm)
    ssm = mask_germany(ssm)

    lst = reduce_to_decad(lst)
    lst = align_to_other_cube(ssm, lst)
    lst = mask_germany(lst)

    return gpp, ssm, lst

def download_means(gpp, ssm, lst, german_geometry):
    #print('downloading gpp')
    #gpp_means = get_mean(gpp, spatial_extent).execute()
    #with open('means/gppmeans.json', "w") as file:
    #    json.dump(gpp_means, file, indent=4)
    #print('downloading ssm')
    #ssm_means = get_mean(ssm, spatial_extent).execute()
    #with open('means/ssmmeans.json', "w") as file:
    #    json.dump(ssm_means, file, indent=4)
    print('downloading lst')
    lst_means = get_mean(lst, german_geometry).execute()
    with open('means/lstmeans.json', "w") as file:
        json.dump(lst_means, file, indent=4)
    print('done')

    return lst_means



# gpp.download('gpp.tif')

computing und downloaden

In [48]:
gpp, ssm, lst = load_datacubes()
#gpp, ssm, lst = preprocess_maps(gpp, ssm, lst)
german_geometry = get_german_geometry()
lst_means = download_means(gpp, ssm, lst, german_geometry)

downloading lst
done


import json means

In [51]:
with open('means/gppmeans.json', 'r') as file:
    gpp_means = json.load(file)
with open('means/gppmeans.json', 'r') as file:
    ssm_means = json.load(file)
with open('means/lstmeans.json', 'r') as file:
    lst_means = json.load(file)

In [53]:
def extract_data(data): 
    dates = [datetime.fromisoformat(date.replace("Z", "+00:00")) for date in data] 
    values = [value[0][0] for value in data.values()] 
    return dates, values 

gpp_dates, gpp_values = extract_data(gpp_means) 
ssm_dates, ssm_values = extract_data(ssm_means) 
lst_dates, lst_values = extract_data(lst_means)

plt.plot(gpp_dates, gpp_values, label="GPP") 
plt.plot(ssm_dates, ssm_values, label="SSM") 
plt.plot(lst_dates, lst_values, label="LST") 
plt.xlabel("Date") 
plt.ylabel("Mean value") 
plt.legend() 
plt.grid() 
plt.show()

NameError: name 'datetime' is not defined